In [ ]:
import os, subprocess
os.makedirs("build", exist_ok=True)

In [ ]:
# --- WaveDrom / VCD rendering utilities (auto-generated — do not edit) ---
import json, re, subprocess
from IPython.display import HTML, display

def _vcd_to_wavedrom(vcd_path, max_cycles=80, signals=None):
    """
    Minimal VCD → WaveDrom JSON converter.
    Works for single-bit and multi-bit signals from iverilog output.
    """
    with open(vcd_path) as f:
        vcd = f.read()

    # Parse variable declarations
    var_map = {}  # id_code → (name, width)
    for m in re.finditer(
        r"\$var\s+\w+\s+(\d+)\s+(\S+)\s+(\S+)", vcd
    ):
        width, code, name = int(m.group(1)), m.group(2), m.group(3)
        if signals is None or name in signals:
            var_map[code] = (name, width)

    if not var_map:
        print(f"⚠  No matching signals in {vcd_path}")
        return None

    # Parse value changes
    changes = {}  # id_code → [(time, value), ...]
    current_time = 0
    for line in vcd.splitlines():
        line = line.strip()
        if line.startswith("#"):
            current_time = int(line[1:])
        elif line.startswith("b"):
            parts = line.split()
            if len(parts) == 2 and parts[1] in var_map:
                val = int(parts[0][1:], 2) if parts[0][1:] else 0
                changes.setdefault(parts[1], []).append((current_time, val))
        elif len(line) >= 2 and line[0] in "01xXzZ" and line[1:] in var_map:
            code = line[1:]
            val = {"0": 0, "1": 1, "x": "x", "X": "x", "z": "z", "Z": "z"}[line[0]]
            changes.setdefault(code, []).append((current_time, val))

    # Determine time step (smallest non-zero delta)
    all_times = sorted({t for ch in changes.values() for t, _ in ch})
    if len(all_times) < 2:
        return None
    deltas = [all_times[i+1] - all_times[i] for i in range(len(all_times)-1) if all_times[i+1] != all_times[i]]
    if not deltas:
        return None
    step = min(deltas)
    end_time = min(all_times[-1], step * max_cycles) if max_cycles else all_times[-1]
    sample_times = list(range(0, end_time + 1, step))[:max_cycles]

    # Build WaveDrom signal list
    wave_signals = []
    for code, (name, width) in sorted(var_map.items(), key=lambda x: x[1][0]):
        ch = sorted(changes.get(code, []))
        # Sample at each time point
        samples = []
        cur_val = 0
        ci = 0
        for t in sample_times:
            while ci < len(ch) and ch[ci][0] <= t:
                cur_val = ch[ci][1]
                ci += 1
            samples.append(cur_val)

        if width == 1:
            # Single-bit: WaveDrom wave string
            wave_str = ""
            for v in samples:
                if v == "x":
                    wave_str += "x"
                elif v == "z":
                    wave_str += "z"
                else:
                    wave_str += str(int(v))
            wave_signals.append({"name": name, "wave": wave_str})
        else:
            # Multi-bit: use '=' for data with labels
            wave_str = ""
            data = []
            prev = None
            for v in samples:
                if v == prev:
                    wave_str += "."
                else:
                    wave_str += "="
                    data.append(f"0x{v:X}" if isinstance(v, int) else str(v))
                    prev = v
            wave_signals.append({"name": name, "wave": wave_str, "data": data})

    return {"signal": wave_signals, "config": {"hscale": 1}}


def show_waves(vcd_path="dump.vcd", max_cycles=80, signals=None, width=900):
    """Render VCD waveforms inline using WaveDrom."""
    wd = _vcd_to_wavedrom(vcd_path, max_cycles=max_cycles, signals=signals)
    if wd is None:
        print("No waveform data to display.")
        return
    wd_json = json.dumps(wd)
    html = f"""
    <div id="wd_{id(wd):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wd):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wd):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))


def show_wavedrom(wave_dict, width=900):
    """Render a raw WaveDrom JSON dict inline."""
    wd_json = json.dumps(wave_dict)
    html = f"""
    <div id="wd_{id(wave_dict):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wave_dict):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wave_dict):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))

print("✓ WaveDrom helpers loaded — use show_waves('dump.vcd') after simulation")

# Day 9 Lab: Memory — RAM, ROM & Block RAM

## Overview
Today you'll work with on-chip memory: ROM for lookup tables and pattern
storage, RAM for read/write data, and the iCE40's block RAM (EBR) resources.
You'll learn to initialize memory from `.hex` files and write testbenches that
verify memory operations with proper handling of synchronous read latency.

## Prerequisites
- Completed Day 8 lab (hierarchical design)
- Pre-class video on ROM, RAM, and iCE40 memory resources watched

## Exercises

| # | Exercise | Time | Key SLOs |
|---|----------|------|----------|
| 1 | ROM Pattern Sequencer | 30 min | 9.1, 9.5, 9.6 |
| 2 | Synchronous RAM — Write/Read | 30 min | 9.2, 9.3, 9.4 |
| 3 | Initialized RAM with `$readmemh` | 25 min | 9.2, 9.6 |
| 4 | Dual-Display Pattern Player (stretch) | 20 min | 9.5, 9.6 |
| 5 | Register File (stretch) | 20 min | 9.2 |

## Key Concepts
- `case`-based ROM vs. array + `$readmemh` ROM
- Async read → LUT inference. Sync read → block RAM inference
- Single-port synchronous RAM with read-before-write behavior
- iCE40 HX1K: 16 EBR blocks × 4 Kbit = 64 Kbit total block RAM
- One-cycle read latency is the #1 source of memory bugs

## Deliverables
- [ ] ROM pattern sequencer displaying on LEDs and 7-seg (Ex 1)
- [ ] RAM write/read verified with self-checking testbench (Ex 2)
- [ ] Initialized RAM with `.hex` file verified (Ex 3)
- [ ] `make stat` output showing block RAM inference for Ex 2

---
## Exercise Files

The starter files for each exercise are below. Edit the code, then run the simulation/build cells.

#### 📝 Exercise 1 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile pattern_sequencer.v
// pattern_sequencer.v — ROM-based LED + 7-seg pattern player
// Loads 16 × 8-bit patterns from patterns.hex
// Lower 4 bits → LEDs, upper 4 bits → 7-seg hex digit
//
// Modes: manual advance (button press) or auto-advance (configurable rate)
module pattern_sequencer #(
    parameter CLK_FREQ     = 25_000_000,
    parameter N_PATTERNS   = 16,
    parameter AUTO_RATE_HZ = 2
)(
    input  wire       i_clk,
    input  wire       i_reset,     // synchronous reset
    input  wire       i_next,      // manual advance (single pulse)
    input  wire       i_auto_mode, // 0 = manual, 1 = auto-advance
    output wire [7:0] o_pattern,   // current pattern word
    output wire [3:0] o_index      // current pattern index
);

    // ────────────────────────────────────────────
    // ROM — loaded from hex file
    // ────────────────────────────────────────────
    reg [7:0] r_patterns [0:N_PATTERNS-1];
    initial $readmemh("patterns.hex", r_patterns);

    // ────────────────────────────────────────────
    // Auto-advance tick generator
    // ────────────────────────────────────────────
    localparam TICK_COUNT = CLK_FREQ / AUTO_RATE_HZ;
    reg [$clog2(TICK_COUNT)-1:0] r_tick_counter;
    wire w_auto_tick;

    // TODO: Implement the tick counter
    //       Count from 0 to TICK_COUNT-1, then wrap
    //       w_auto_tick should pulse high for one cycle when count wraps
    //       Reset the counter on i_reset
    //
    // ---- YOUR CODE HERE ----

    // ────────────────────────────────────────────
    // Advance selection: auto tick or manual button
    // ────────────────────────────────────────────
    wire w_advance = i_auto_mode ? w_auto_tick : i_next;

    // ────────────────────────────────────────────
    // Address counter
    // ────────────────────────────────────────────
    reg [3:0] r_addr;

    // TODO: Implement the address counter
    //       On i_reset: r_addr <= 0
    //       On w_advance: r_addr <= (r_addr == N_PATTERNS-1) ? 0 : r_addr + 1
    //
    // ---- YOUR CODE HERE ----

    // ────────────────────────────────────────────
    // ROM read (async — fine for 16-entry LUT ROM)
    // ────────────────────────────────────────────
    assign o_pattern = r_patterns[r_addr];
    assign o_index   = r_addr;

endmodule

In [ ]:
%%writefile patterns.hex
01
12
24
38
4C
5A
69
75
8F
9E
AD
BB
C7
D3
E1
F0

In [ ]:
%%writefile tb_pattern_sequencer.v
// tb_pattern_sequencer.v — Testbench for ROM pattern sequencer
`timescale 1ns / 1ps

module tb_pattern_sequencer;

    reg        clk, reset, next, auto_mode;
    wire [7:0] pattern;
    wire [3:0] index;

    // Fast simulation: CLK_FREQ=100, AUTO_RATE_HZ=10 → 10 clocks per tick
    pattern_sequencer #(
        .CLK_FREQ     (100),
        .N_PATTERNS   (16),
        .AUTO_RATE_HZ (10)
    ) uut (
        .i_clk       (clk),
        .i_reset     (reset),
        .i_next      (next),
        .i_auto_mode (auto_mode),
        .o_pattern   (pattern),
        .o_index     (index)
    );

    initial clk = 0;
    always #5 clk = ~clk;  // 100 MHz sim clock

    integer test_count = 0, fail_count = 0;

    task check(input [3:0] exp_idx, input [7:0] exp_pat, input [159:0] msg);
        begin
            test_count = test_count + 1;
            if (index !== exp_idx || pattern !== exp_pat) begin
                fail_count = fail_count + 1;
                $display("FAIL [%0s]: idx=%0d (exp %0d) pat=0x%02X (exp 0x%02X)",
                         msg, index, exp_idx, pattern, exp_pat);
            end
        end
    endtask

    initial begin
        $dumpfile("pattern_seq.vcd");
        $dumpvars(0, tb_pattern_sequencer);

        reset = 1; next = 0; auto_mode = 0;
        @(posedge clk); @(posedge clk);
        reset = 0;
        @(posedge clk); #1;

        // ── Test 1: Reset state ──
        // TODO: Verify index=0, pattern=0x01 after reset
        //       Use: check(4'd0, 8'h01, "reset state");
        //
        // ---- YOUR CODE HERE ----

        // ── Test 2: Manual advance through all 16 patterns ──
        // TODO: Pulse i_next for one clock, then check each pattern
        //       Expected sequence from patterns.hex:
        //       idx 0=0x01, 1=0x12, 2=0x24, 3=0x38, ...
        //       After idx 15 (0xF0), should wrap to idx 0 (0x01)
        //
        // ---- YOUR CODE HERE ----

        // ── Test 3: Auto-advance mode ──
        // TODO: Set auto_mode=1, wait for tick period (10 clocks),
        //       verify pattern advances automatically.
        //       Run for several ticks and check at least 3 advances.
        //
        // ---- YOUR CODE HERE ----

        // ── Report ──
        $display("\n=================================");
        if (fail_count == 0)
            $display("ALL %0d TESTS PASSED", test_count);
        else
            $display("FAILED %0d / %0d tests", fail_count, test_count);
        $display("=================================\n");
        $finish;
    end
endmodule

In [ ]:
%%writefile Makefile
# Makefile — pattern_sequencer
PROJECT  = pattern_sequencer
TOP      = pattern_sequencer
PCF      = ../go_board.pcf
SRCS     = $(wildcard *.v)
TB       = tb_pattern_sequencer.v

DEVICE   = hx1k
PACKAGE  = vq100

IVERILOG_FLAGS = -g2012 -Wall

all: sim

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(filter-out $(TB),$(SRCS))
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all prog sim wave stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_pattern_sequencer.v pattern_sequencer.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

#### 📝 Exercise 2 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ram_sp.v
// ram_sp.v — Single-port synchronous RAM
// Synchronous read → will infer block RAM (EBR) on iCE40
// Read-before-write behavior: on simultaneous read+write to same
// address, the read returns the OLD value.
module ram_sp #(
    parameter ADDR_WIDTH = 8,    // 2^8 = 256 addresses
    parameter DATA_WIDTH = 8     // 8-bit data
)(
    input  wire                    i_clk,
    input  wire                    i_write_en,
    input  wire [ADDR_WIDTH-1:0]  i_addr,
    input  wire [DATA_WIDTH-1:0]  i_write_data,
    output reg  [DATA_WIDTH-1:0]  o_read_data
);

    // Memory array
    reg [DATA_WIDTH-1:0] r_mem [0:(1 << ADDR_WIDTH)-1];

    // TODO: Implement synchronous read-before-write behavior
    //
    //       On every rising edge of i_clk:
    //         1. If i_write_en is high, write i_write_data to r_mem[i_addr]
    //         2. Always read r_mem[i_addr] into o_read_data
    //
    //       IMPORTANT: The read happens on the SAME clock edge.
    //       This means the read data appears ONE CYCLE AFTER the
    //       address is presented. This one-cycle latency is what
    //       allows Yosys to infer block RAM instead of LUTs.
    //
    //       Pattern:
    //         always @(posedge i_clk) begin
    //             if (i_write_en)
    //                 r_mem[i_addr] <= i_write_data;
    //             o_read_data <= r_mem[i_addr];
    //         end
    //
    // ---- YOUR CODE HERE ----

endmodule

In [ ]:
%%writefile tb_ram_sp.v
// tb_ram_sp.v — Testbench for single-port synchronous RAM
`timescale 1ns / 1ps

module tb_ram_sp;

    reg        clk, we;
    reg  [7:0] addr, wdata;
    wire [7:0] rdata;

    ram_sp #(.ADDR_WIDTH(8), .DATA_WIDTH(8)) uut (
        .i_clk        (clk),
        .i_write_en   (we),
        .i_addr       (addr),
        .i_write_data (wdata),
        .o_read_data  (rdata)
    );

    initial clk = 0;
    always #20 clk = ~clk;  // 25 MHz

    integer i, test_count = 0, fail_count = 0;

    task check_read(input [7:0] exp, input [255:0] msg);
        begin
            test_count = test_count + 1;
            if (rdata !== exp) begin
                fail_count = fail_count + 1;
                $display("FAIL [%0s]: got 0x%02X, expected 0x%02X",
                         msg, rdata, exp);
            end
        end
    endtask

    initial begin
        $dumpfile("ram.vcd");
        $dumpvars(0, tb_ram_sp);
        we = 0; addr = 0; wdata = 0;
        #100;

        // ── Test 1: Write XOR pattern to all 256 addresses ──
        // TODO: Set we=1, loop i from 0 to 255
        //       addr = i, wdata = i ^ 8'hA5
        //       One @(posedge clk); #1; per iteration
        //       Then set we=0
        //
        // ---- YOUR CODE HERE ----

        // ── Test 2: Read back and verify all 256 addresses ──
        // TODO: Loop i from 0 to 255
        //       Set addr = i
        //       Wait @(posedge clk); #1;  <-- THIS IS THE KEY!
        //       One-cycle read latency means data appears NEXT cycle.
        //       Use check_read(i ^ 8'hA5, "readback") to verify
        //
        // ---- YOUR CODE HERE ----

        // ── Test 3: Overwrite and verify ──
        // TODO: Write a different pattern (e.g., i ^ 8'h5A) to
        //       addresses 0-15 only, then verify those changed
        //       while addresses 16+ still have old pattern.
        //
        // ---- YOUR CODE HERE ----

        // Report
        $display("\n=================================");
        if (fail_count == 0)
            $display("ALL %0d TESTS PASSED", test_count);
        else
            $display("FAILED %0d / %0d tests", fail_count, test_count);
        $display("=================================\n");
        $finish;
    end
endmodule

In [ ]:
%%writefile Makefile
# Makefile — ram_sp
PROJECT  = ram_sp
TOP      = ram_sp
PCF      = ../go_board.pcf
SRCS     = $(wildcard *.v)
TB       = tb_ram_sp.v

DEVICE   = hx1k
PACKAGE  = vq100

IVERILOG_FLAGS = -g2012 -Wall

all: sim

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(filter-out $(TB),$(SRCS))
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all prog sim wave stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_ram_sp.v ram_sp.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

#### 📝 Exercise 3 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile init_data.hex
00
20
40
60
80
A0
C0
E0
FF
E0
C0
A0
80
60
40
20

In [ ]:
%%writefile ram_init.v
// ram_init.v — Synchronous RAM with $readmemh initialization
// Combines ROM (initial data) + RAM (writeable) in one module.
// After programming, the RAM contains the hex file's data.
// Writes update individual addresses without affecting others.
module ram_init #(
    parameter ADDR_WIDTH = 4,     // 2^4 = 16 addresses
    parameter DATA_WIDTH = 8,
    parameter INIT_FILE  = "init_data.hex"
)(
    input  wire                    i_clk,
    input  wire                    i_write_en,
    input  wire [ADDR_WIDTH-1:0]  i_addr,
    input  wire [DATA_WIDTH-1:0]  i_write_data,
    output reg  [DATA_WIDTH-1:0]  o_read_data
);

    reg [DATA_WIDTH-1:0] r_mem [0:(1 << ADDR_WIDTH)-1];

    // TODO: Load initial contents from hex file
    //       Use: initial $readmemh(INIT_FILE, r_mem);
    //
    // ---- YOUR CODE HERE ----

    // TODO: Implement synchronous read-before-write (same as ram_sp)
    //
    // ---- YOUR CODE HERE ----

endmodule

In [ ]:
%%writefile tb_ram_init.v
// tb_ram_init.v — Testbench for initialized RAM
`timescale 1ns / 1ps

module tb_ram_init;

    reg        clk, we;
    reg  [3:0] addr;
    reg  [7:0] wdata;
    wire [7:0] rdata;

    ram_init #(
        .ADDR_WIDTH (4),
        .DATA_WIDTH (8),
        .INIT_FILE  ("init_data.hex")
    ) uut (
        .i_clk        (clk),
        .i_write_en   (we),
        .i_addr       (addr),
        .i_write_data (wdata),
        .o_read_data  (rdata)
    );

    initial clk = 0;
    always #20 clk = ~clk;

    integer i, test_count = 0, fail_count = 0;

    // Expected initial data (triangular wave from init_data.hex)
    reg [7:0] init_vals [0:15];
    initial begin
        init_vals[0]  = 8'h00; init_vals[1]  = 8'h20;
        init_vals[2]  = 8'h40; init_vals[3]  = 8'h60;
        init_vals[4]  = 8'h80; init_vals[5]  = 8'hA0;
        init_vals[6]  = 8'hC0; init_vals[7]  = 8'hE0;
        init_vals[8]  = 8'hFF; init_vals[9]  = 8'hE0;
        init_vals[10] = 8'hC0; init_vals[11] = 8'hA0;
        init_vals[12] = 8'h80; init_vals[13] = 8'h60;
        init_vals[14] = 8'h40; init_vals[15] = 8'h20;
    end

    task check(input [7:0] exp, input [255:0] msg);
        begin
            test_count = test_count + 1;
            if (rdata !== exp) begin
                fail_count = fail_count + 1;
                $display("FAIL [%0s]: addr=%0d got 0x%02X, expected 0x%02X",
                         msg, addr, rdata, exp);
            end
        end
    endtask

    initial begin
        $dumpfile("ram_init.vcd");
        $dumpvars(0, tb_ram_init);
        we = 0; addr = 0; wdata = 0;
        #100;

        // TODO: Test 1 — Verify all 16 initial values
        //       Loop through addresses, read each, compare to init_vals[]
        //       Remember the one-cycle read latency!
        //
        // ---- YOUR CODE HERE ----

        // TODO: Test 2 — Write new values to addresses 2 and 5
        //       addr=2, wdata=8'hBE; addr=5, wdata=8'hEF
        //
        // ---- YOUR CODE HERE ----

        // TODO: Test 3 — Verify writes took effect at addr 2 and 5
        //       AND verify untouched addresses still hold initial values
        //       (Check at least addresses 0, 1, 3, 4 are unchanged)
        //
        // ---- YOUR CODE HERE ----

        $display("\n=================================");
        if (fail_count == 0)
            $display("ALL %0d TESTS PASSED", test_count);
        else
            $display("FAILED %0d / %0d tests", fail_count, test_count);
        $display("=================================\n");
        $finish;
    end
endmodule

In [ ]:
%%writefile Makefile
# Makefile — ram_init
PROJECT  = ram_init
TOP      = ram_init
PCF      = ../go_board.pcf
SRCS     = $(wildcard *.v)
TB       = tb_ram_init.v

DEVICE   = hx1k
PACKAGE  = vq100

IVERILOG_FLAGS = -g2012 -Wall

all: sim

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(filter-out $(TB),$(SRCS))
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all prog sim wave stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_ram_init.v ram_init.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

#### 📝 Exercise 4 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile display1_patterns.hex
00
01
02
03
04
05
06
07
08
09
0A
0B
0C
0D
0E
0F

In [ ]:
%%writefile display2_patterns.hex
0F
0E
0D
0C
0B
0A
09
08
07
06
05
04
03
02
01
00

In [ ]:
%%writefile dual_rom.v
// =============================================================================
// dual_rom.v — Two Independent ROMs with Shared Address Counter (Starter)
// Day 9, Exercise 4 (Stretch)
// =============================================================================
// Demonstrates: multiple ROM instances, shared addressing, speed control
//
// TODO: Implement a module with:
//   - Two ROMs loaded from hex files (display1_patterns.hex, display2_patterns.hex)
//   - A shared address counter that cycles through entries
//   - A speed selector (button press cycles through 1 Hz, 2 Hz, 4 Hz)
//   - Outputs: o_data1[7:0], o_data2[7:0], o_addr[3:0]

module dual_rom #(
    parameter CLK_FREQ  = 25_000_000,
    parameter N_ENTRIES = 16
)(
    input  wire       i_clk,
    input  wire       i_reset,
    input  wire       i_speed_up,
    output wire [7:0] o_data1,
    output wire [7:0] o_data2,
    output wire [3:0] o_addr
);

    // ROM arrays
    reg [7:0] r_rom1 [0:N_ENTRIES-1];
    reg [7:0] r_rom2 [0:N_ENTRIES-1];
    initial $readmemh("display1_patterns.hex", r_rom1);
    initial $readmemh("display2_patterns.hex", r_rom2);

    // TODO: Speed selection — cycle through 3 rates on i_speed_up pulse

    // TODO: Address counter — increment at the selected rate

    // TODO: ROM outputs — registered reads from both ROMs

endmodule

In [ ]:
%%writefile Makefile
# Makefile — dual_rom
PROJECT  = dual_rom
TOP      = dual_rom
PCF      = ../go_board.pcf
SRCS     = $(wildcard *.v)
TB       = tb_dual_rom.v

DEVICE   = hx1k
PACKAGE  = vq100

IVERILOG_FLAGS = -g2012 -Wall

all: sim

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(filter-out $(TB),$(SRCS))
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all prog sim wave stat clean

#### 📝 Exercise 5 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile register_file.v
// =============================================================================
// register_file.v — Small Register File (Starter)
// Day 9, Exercise 5 (Stretch)
// =============================================================================
// 1 write port (synchronous), 2 read ports (combinational/async)
// Async reads infer distributed RAM (LUTs), not block RAM.
//
// TODO: Implement the register file with:
//   - Parameterized number of registers and data width
//   - Synchronous write: on posedge clk, if i_write_en, write i_write_data to i_write_addr
//   - Async read: o_read_data_a = regs[i_read_addr_a] (combinational)

module register_file #(
    parameter N_REGS     = 8,
    parameter DATA_WIDTH = 8
)(
    input  wire                          i_clk,
    input  wire                          i_write_en,
    input  wire [$clog2(N_REGS)-1:0]    i_write_addr,
    input  wire [DATA_WIDTH-1:0]         i_write_data,
    input  wire [$clog2(N_REGS)-1:0]    i_read_addr_a,
    input  wire [$clog2(N_REGS)-1:0]    i_read_addr_b,
    output wire [DATA_WIDTH-1:0]         o_read_data_a,
    output wire [DATA_WIDTH-1:0]         o_read_data_b
);

    // TODO: Declare the register array

    // TODO: Synchronous write

    // TODO: Async (combinational) reads

endmodule

In [ ]:
%%writefile tb_register_file.v
// tb_register_file.v — Testbench for register file (SOLUTION)
`timescale 1ns / 1ps

module tb_register_file;

    reg        clk, we;
    reg  [2:0] waddr, raddr_a, raddr_b;
    reg  [7:0] wdata;
    wire [7:0] rdata_a, rdata_b;

    register_file #(.N_REGS(8), .DATA_WIDTH(8)) uut (
        .i_clk         (clk),
        .i_write_en    (we),
        .i_write_addr  (waddr),
        .i_write_data  (wdata),
        .i_read_addr_a (raddr_a),
        .i_read_addr_b (raddr_b),
        .o_read_data_a (rdata_a),
        .o_read_data_b (rdata_b)
    );

    initial clk = 0;
    always #20 clk = ~clk;

    integer i, test_count = 0, fail_count = 0;

    task check_ab(input [7:0] exp_a, exp_b, input [255:0] msg);
        begin
            test_count = test_count + 1;
            if (rdata_a !== exp_a || rdata_b !== exp_b) begin
                fail_count = fail_count + 1;
                $display("FAIL [%0s]: A=0x%02X(exp 0x%02X) B=0x%02X(exp 0x%02X)",
                         msg, rdata_a, exp_a, rdata_b, exp_b);
            end else
                $display("PASS [%0s]", msg);
        end
    endtask

    initial begin
        $dumpfile("regfile.vcd");
        $dumpvars(0, tb_register_file);
        we = 0; waddr = 0; wdata = 0; raddr_a = 0; raddr_b = 0;
        #100;

        // Write unique values to all 8 registers
        $display("--- Writing all registers ---");
        we = 1;
        for (i = 0; i < 8; i = i + 1) begin
            waddr = i[2:0];
            wdata = (i * 16 + 5);   // 0x05, 0x15, 0x25, ...
            @(posedge clk); #1;
        end
        we = 0;

        // Read pairs and verify (async — no latency)
        $display("\n--- Dual-port read verification ---");
        raddr_a = 3'd0; raddr_b = 3'd7; #1;
        check_ab(8'h05, 8'h75, "regs 0,7");

        raddr_a = 3'd3; raddr_b = 3'd4; #1;
        check_ab(8'h35, 8'h45, "regs 3,4");

        raddr_a = 3'd1; raddr_b = 3'd1; #1;
        check_ab(8'h15, 8'h15, "same reg both ports");

        // Write-then-read same address (forwarding test)
        $display("\n--- Write-then-read ---");
        we = 1; waddr = 3'd2; wdata = 8'hFF;
        raddr_a = 3'd2; raddr_b = 3'd5;
        @(posedge clk); #1;
        we = 0;
        // After the clock edge, reg 2 should have 8'hFF
        // (async read sees the NEWLY written value on next #1 settling)
        raddr_a = 3'd2; #1;
        check_ab(8'hFF, 8'h55, "write-then-read");

        $display("\n=================================");
        if (fail_count == 0)
            $display("ALL %0d TESTS PASSED", test_count);
        else
            $display("FAILED %0d / %0d tests", fail_count, test_count);
        $display("=================================\n");
        $finish;
    end
endmodule

In [ ]:
%%writefile Makefile
# Makefile — register_file
PROJECT  = register_file
TOP      = register_file
PCF      = ../go_board.pcf
SRCS     = $(wildcard *.v)
TB       = tb_register_file.v

DEVICE   = hx1k
PACKAGE  = vq100

IVERILOG_FLAGS = -g2012 -Wall

all: sim

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(filter-out $(TB),$(SRCS))
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all prog sim wave stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_register_file.v register_file.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')